# Stage 3 dataset creation

This notebook builds the reviewed English knowledge corpus from small, readable seed files. It uses controlled rules for retrieval text and aliases, keeps source IDs, and exports held-out evaluation queries separately.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'PROJECT_PLAN.md').exists():
    project_root = project_root.parent
if not (project_root / 'PROJECT_PLAN.md').exists():
    raise RuntimeError('Run this notebook from the repository or its notebooks directory.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.dataset_builder import DATASET_VERSION, build_dataset
from src.dataset_io import read_json, read_jsonl
print('Repository:', project_root)
print('Dataset version:', DATASET_VERSION)

## Load reviewed seeds

The files separate subject mappings, topic mappings, evidence-backed strategies, planner templates, sensor safety responses, and low-risk alias rules.

In [ ]:
raw_dir = project_root / 'data' / 'raw'
seed_names = [
    'subject_seed_data.json', 'topic_seed_data.json', 'study_strategy_seed.json',
    'session_template_seed.json', 'sensor_intervention_seed.json', 'alias_seed_data.json',
]
seeds = {name: read_json(raw_dir / name) for name in seed_names}
print('Loaded seed files:', len(seeds))
print('Subject groups:', len(seeds['subject_seed_data.json']['subject_groups']))
print('Topic groups:', len(seeds['topic_seed_data.json']['topic_groups']))
print('Strategies:', len(seeds['study_strategy_seed.json']))
print('Session templates:', len(seeds['session_template_seed.json']))
print('Sensor interventions:', len(seeds['sensor_intervention_seed.json']))

## Build and export

The reusable builder creates stable IDs, consistent retrieval text, traced aliases, family JSONL files, the combined corpus, statistics, and a separate evaluation file.

In [ ]:
statistics = build_dataset(project_root)
for family, count in statistics['document_family_counts'].items():
    print(f'{family:22} {count:4}')
print('Canonical:', statistics['canonical_record_count'])
print('Synthetic:', statistics['synthetic_record_count'])
print('Total retrievable:', statistics['retrievable_record_count'])
print('Held-out queries:', statistics['evaluation_query_count'])

## Inspect representative records

In [ ]:
corpus = read_jsonl(project_root / 'data' / 'processed' / 'knowledge_corpus.jsonl')
shown = set()
for record in corpus:
    family = record['document_family']
    if family not in shown:
        shown.add(family)
        print(f'\n[{family}] {record["record_id"]}')
        print(record['retrieval_text'][:320] + ('...' if len(record['retrieval_text']) > 320 else ''))